# 03 - Bronze: UN Comtrade HS6 Bilateral Extract

Ingests `cemac_ecowas_comtrade_annual_hs6_1990_2024.jsonl` into `bronze.comtrade_hs6_raw`.

Each envelope row in the JSONL wraps one reporter×year API response.
This notebook explodes the `payload` array so that the bronze table holds one row
per reporter × partner × HS6 commodity code × flow × year.

**Bronze rule: preserve all source rows.** Aggregate/subtotal rows (`is_aggregate = true`)
are kept here. The silver notebook (`10b`) filters them. If Comtrade changes its
aggregate structure in a future release, bronze still holds the original data.

**Schema after explosion:**

| Column | Description |
|--------|-------------|
| `reporter_iso3` | ISO 3166-1 alpha-3 of the reporting country |
| `partner_iso3` | ISO 3166-1 alpha-3 of the trade partner (`W00` = World aggregate) |
| `partner_name` | Partner label from Comtrade |
| `cmd_code` | HS6 commodity code (6-digit) |
| `cmd_desc` | Commodity description (may be null for older years) |
| `flow` | `X` = exports, `M` = imports |
| `year` | Calendar year |
| `value_usd` | Trade value (primary; CIF for imports, FOB for exports) |
| `fob_value_usd` | FOB value if separately reported |
| `cif_value_usd` | CIF value if separately reported |
| `net_wgt_kg` | Net weight |
| `is_reported` | True if country directly reported; False if imputed from mirror |
| `is_aggregate` | True if row is a subtotal — **preserved in bronze, filtered in silver** |
| `classification` | HS revision (`H0`–`H6`) |
| `source_api` | Fixed: `UN_COMTRADE_V3` |
| `extraction_run_id` | Timestamp of the notebook run that loaded the row |
| `extracted_at` | Timestamp of original API call |
| `loaded_at` | Timestamp of this notebook run |


In [ ]:
# Cell 1 - Configuration
from datetime import datetime, timezone

spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

JSONL_PATH = "/Volumes/cemac_ecowas_aes_trade/bronze/raw_landing/comtrade/cemac_ecowas_comtrade_annual_hs6_1990_2024.jsonl"
TARGET_TABLE = "bronze.comtrade_hs6_raw"
loaded_at = datetime.now(timezone.utc)

print(f"Source : {JSONL_PATH}")
print(f"Target : {TARGET_TABLE}")
print(f"Loaded : {loaded_at}")

In [ ]:
# Cell 2 - Read and explode JSONL envelopes
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType, BooleanType, DoubleType, IntegerType,
    StringType, StructField, StructType,
)

payload_schema = ArrayType(
    StructType([
        StructField("reporter_iso3",  StringType(),  True),
        StructField("partner_code",   IntegerType(), True),
        StructField("partner_iso3",   StringType(),  True),
        StructField("partner_name",   StringType(),  True),
        StructField("cmd_code",       StringType(),  True),
        StructField("cmd_desc",       StringType(),  True),
        StructField("flow",           StringType(),  True),
        StructField("period",         StringType(),  True),
        StructField("value_usd",      DoubleType(),  True),
        StructField("cif_value_usd",  DoubleType(),  True),
        StructField("fob_value_usd",  DoubleType(),  True),
        StructField("net_wgt_kg",     DoubleType(),  True),
        StructField("gross_wgt_kg",   DoubleType(),  True),
        StructField("qty",            DoubleType(),  True),
        StructField("qty_unit",       StringType(),  True),
        StructField("is_reported",    BooleanType(), True),
        StructField("is_aggregate",   BooleanType(), True),
        StructField("is_intra_bloc",  BooleanType(), True),
        StructField("classification", StringType(),  True),
    ])
)

envelope_df = (
    spark.read.json(JSONL_PATH)
    .withColumn("payload", F.from_json(F.col("payload").cast("string"), payload_schema))
)

print(f"Envelopes loaded: {envelope_df.count():,}")

# Count envelopes with empty payload
empty = envelope_df.where(F.size(F.col("payload")) == 0).count()
print(f"Empty envelopes (countries/years with no HS6 submission): {empty:,}")

In [ ]:
# Cell 3 - Explode payload, normalise, and add audit columns
# Bronze rule: preserve all rows — is_aggregate rows are kept here; silver filters them.
raw_df = (
    envelope_df
    .where(F.size(F.col("payload")) > 0)
    .select(
        F.col("extracted_at"),
        F.explode(F.col("payload")).alias("r"),
    )
    .select(
        F.upper(F.trim(F.col("r.reporter_iso3"))).alias("reporter_iso3"),
        F.upper(F.trim(F.col("r.partner_iso3"))).alias("partner_iso3"),
        F.col("r.partner_name").alias("partner_name"),
        F.col("r.cmd_code").alias("cmd_code"),
        F.col("r.cmd_desc").alias("cmd_desc"),
        F.upper(F.trim(F.col("r.flow"))).alias("flow"),
        F.col("r.period").cast("int").alias("year"),
        F.col("r.value_usd").alias("value_usd"),
        F.col("r.fob_value_usd").alias("fob_value_usd"),
        F.col("r.cif_value_usd").alias("cif_value_usd"),
        F.col("r.net_wgt_kg").alias("net_wgt_kg"),
        F.col("r.qty").alias("qty"),
        F.col("r.qty_unit").alias("qty_unit"),
        F.col("r.is_reported").alias("is_reported"),
        F.col("r.is_aggregate").alias("is_aggregate"),
        F.col("r.is_intra_bloc").alias("is_intra_bloc"),
        F.col("r.classification").alias("hs_classification"),
        F.col("extracted_at"),
        F.lit("UN_COMTRADE_V3").alias("source_api"),
        F.lit(str(loaded_at)).alias("extraction_run_id"),
        F.lit(str(loaded_at)).cast("timestamp").alias("loaded_at"),
    )
    # Minimal null guards only — bronze does NOT filter is_aggregate rows.
    .where(F.col("reporter_iso3").isNotNull())
    .where(F.col("partner_iso3").isNotNull())
    .where(F.col("flow").isin(["X", "M"]))
)

print(f"Trade line rows after explode: {raw_df.count():,}")
raw_df.groupBy("flow").count().orderBy("flow").show()

# Show aggregate vs non-aggregate breakdown (not filtered — both are kept in bronze)
print("Aggregate vs non-aggregate row counts:")
raw_df.groupBy("is_aggregate").count().show()

raw_df.show(5, truncate=False)


In [ ]:
# Cell 4 - Coverage diagnostics before writing
print("Reporter coverage (all rows, including aggregates):")
raw_df.groupBy("reporter_iso3").agg(
    F.count("*").alias("total_rows"),
    F.sum(F.when(~F.coalesce(F.col("is_aggregate"), F.lit(False)), F.lit(1)).otherwise(F.lit(0))).alias("hs6_line_rows"),
    F.sum(F.when(F.coalesce(F.col("is_aggregate"), F.lit(False)), F.lit(1)).otherwise(F.lit(0))).alias("aggregate_rows"),
    F.countDistinct(F.when(~F.coalesce(F.col("is_aggregate"), F.lit(False)), F.col("partner_iso3"))).alias("bilateral_partners"),
    F.countDistinct(F.when(~F.coalesce(F.col("is_aggregate"), F.lit(False)), F.col("cmd_code"))).alias("hs6_codes"),
    F.min("year").alias("first_year"),
    F.max("year").alias("last_year"),
).orderBy(F.desc("hs6_line_rows")).show(25, truncate=False)

print("Rows by year (all reporters combined, non-aggregate only):")
raw_df.where(~F.coalesce(F.col("is_aggregate"), F.lit(False))).groupBy("year").count().orderBy("year").show(40)


In [ ]:
# Cell 5 - Write bronze.comtrade_hs6_raw
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")

(
    raw_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("reporter_iso3", "year")
    .saveAsTable(TARGET_TABLE)
)

print(f"Write complete: {TARGET_TABLE}")
spark.sql(f"SELECT COUNT(*) AS rows FROM {TARGET_TABLE}").show()
spark.sql(f"DESCRIBE TABLE {TARGET_TABLE}").show(40, truncate=False)